### Problem Statement

#### The decision problem is:

#### Does a simple card-counting-style betting strategy perform better than flat betting, and how does that affect profit and risk?

#### We compare three types of players:

* No-strategy player: flat bet every hand, worse expected value.
* Basic-strategy player: flat bet every hand, slightly better expected value.
* Simple card counter: uses a simplified count signal and changes bet size depending on whether the table is favorable.
#### Key uncertainties

The main uncertainty is the outcome of each blackjack hand. Even a strong player can lose many hands in a row.

#### Key trade-offs

* betting more when the count is favorable may increase expected profit,
* but larger bets also increase volatility and the chance of short-run losses.
Objective

The objective is to estimate:

* average profit
* profit variability
* probability of finishing with a loss
* probability of ruin

### Inputs, Parametrization, and Assumptions

#### Inputs used:

* Number of sessions simulated
* Number of hands per session
* Initial bankroll
* Flat or variable bet sizes
* Player edge by strategy
* Probability that the count is favorable for the counter
  
#### Source-based elements

* compare random/no-strategy, basic strategy, and card counting
* use blackjack as a risk-management simulation setting
* analyze EV, variance, bankroll ruin, and profit distribution
* use low and high bet sizes
 
#### Assumptions

* No-strategy player expected edge = -2.0%
* Basic-strategy player expected edge = -0.5%
* Counter expected edge:
    * -0.5% when the count is unfavorable
    * +1.0% when the count is favorable
* Probability of favorable count = 30%
* Low bet = 10
* High bet = 50
* Starting bankroll = 1,000
* Session length = 200 hands

In [ ]:
# Import Necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Summary statistics function

def summarystats(df):
    summary = pd.DataFrame({
        "Mean": df.mean(),
        "Std Dev": df.std(ddof=1),
        "Min": df.min(),
        "5th Percentile": df.quantile(0.05),
        "Median": df.median(),
        "95th Percentile": df.quantile(0.95),
        "Max": df.max()
    })
    return summary.round(2)

In [ ]:
# Simulate one blackjack hand
def simulate_hand(bet, edge, rng):
    """
    Simplified blackjack outcome model.
    We assume the standard deviation per hand is about 1.15 betting units.
    Then generate a hand profit in betting units from a normal distribution.
    """
    units = rng.normal(loc=edge, scale=1.15)
    profit = bet * units
    return profit


In [ ]:
# Simulate sessions for one player type
def simulate_sessions(player_type, n_sessions=5000, n_hands=200, bankroll0=1000, seed=100):
    rng = np.random.default_rng(seed)
    results = []

    for _ in range(n_sessions):
        bankroll = bankroll0
        ruined = 0

        for _ in range(n_hands):
            if bankroll <= 0:
                ruined = 1
                bankroll = 0
                break

            # Set betting and edge rules
            if player_type == "no_strategy":
                bet = min(25, bankroll)
                edge = -0.02

            elif player_type == "basic":
                bet = min(25, bankroll)
                edge = -0.005

            elif player_type == "counter":
                favorable_count = rng.random() < 0.30  # 30% of hands are favorable
                if favorable_count:
                    bet = min(50, bankroll)
                    edge = 0.01
                else:
                    bet = min(10, bankroll)
                    edge = -0.005
            else:
                raise ValueError("Invalid player type")

            bankroll += simulate_hand(bet, edge, rng)

        results.append({
            "Final Bankroll": bankroll,
            "Profit": bankroll - bankroll0,
            "Loss Indicator": 1 if bankroll < bankroll0 else 0,
            "Ruin Indicator": ruined
        })

    return pd.DataFrame(results)


In [ ]:
# Run simulations
no_strategy_df = simulate_sessions("no_strategy")
basic_df = simulate_sessions("basic")
counter_df = simulate_sessions("counter")

In [ ]:
# Print summary stats
print("\nNo Strategy")
print(summarystats(no_strategy_df))

print("\nBasic Strategy")
print(summarystats(basic_df))

print("\nCound Cards")
print(summarystats(counter_df))

# Probability measures
print("\nProbability of finishing with a loss:")
print("No strategy:", round(no_strategy_df["Loss Indicator"].mean(), 4))
print("Basic:", round(basic_df["Loss Indicator"].mean(), 4))
print("Counter:", round(counter_df["Loss Indicator"].mean(), 4))

print("\nProbability of ruin:")
print("No strategy:", round(no_strategy_df["Ruin Indicator"].mean(), 4))
print("Basic:", round(basic_df["Ruin Indicator"].mean(), 4))
print("Counter:", round(counter_df["Ruin Indicator"].mean(), 4))

In [ ]:
# Histograms
plt.figure(figsize=(8,5))
plt.hist(no_strategy_df["Profit"], bins=40, alpha=0.6, label="No Strategy")
plt.hist(basic_df["Profit"], bins=40, alpha=0.6, label="Basic")
plt.hist(counter_df["Profit"], bins=40, alpha=0.6, label="Counter")
plt.axvline(0, linestyle="--")
plt.xlabel("Session Profit")
plt.ylabel("Frequency")
plt.title("Distribution of Session Profit")
plt.legend()
plt.show()

The simulation results clearly show that strategy choice has a meaningful impact on both expected profit and risk exposure. The no-strategy player performs the worst, with an average loss of about -104 per session and the highest probability of losing money (60.2%). The basic strategy player performs better, reducing the average loss to about -29 per session and lowering the probability of loss to 52.6%. The card counter is the only strategy with a positive expected profit, averaging about +29 per session, and has the lowest probability of losing money (47.2%). This demonstrates that improved decision-making (especially card counting) can shift the expected value from negative to positive.

The histogram shows that all three strategies have wide and overlapping distributions, meaning outcomes are highly uncertain in the short run. The card counter’s distribution is shifted to the right, indicating higher expected profits. However, it is also more spread out, reflecting greater variability due to larger bets in favorable situations. The no-strategy and basic players have distributions centered below zero, confirming consistent long-run losses. This highlights a key risk management insight: even when a strategy has a positive expected value, individual outcomes can vary widely.

Looking at the 5th percentile, no strategy: about -807, basic strategy: about -714, and counter: about -729. All three strategies experience significant downside risk, and the counter does not eliminate losses, despite having a positive average return. Looking at Probability of Loss,
no strategy: 60.2%, basic strategy: 52.6%, and counter: 47.2%. Even the card counter loses money in nearly half of all sessions, reinforcing that positive expected value does not guarantee short-term success. Looking at Probability of Ruin, no strategy: 2.78%, basic strategy: 1.68%, and Counter: 2.18%. Interestingly, the counter has a higher probability of ruin than the basic strategy player, despite having a higher expected profit. This occurs because the counter places larger bets, increasing volatility and higher volatility increases the chance of hitting zero bankroll, even with a positive edge. Risk tolerance matters - A risk-averse decision-maker might prefer basic strategy (lower ruin probability) and a risk-seeking decision-maker might prefer card counting (positive expected return).

Card counting improves expected profitability but introduces greater volatility, meaning that while it is optimal in the long run, it still carries meaningful short-term risk and potential for losses.